## RANDOM FOREST REGRESSION

## Input: Bel, Awa, epi, CHI----> Output: BEH

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

df= pd.read_csv('FULL_DATA5_FINALE.csv')  # recall to set path

dl=df[['pct_mask',
      'pct_worried_catch_covid','pct_belief_masking_effective',
     'pct_received_news_local_health','pct_received_news_experts','pct_received_news_who','pct_received_news_govt_health' ,'pct_received_news_politicians','pct_received_news_journalists','pct_received_news_friends',
     'new_cases_smoothed','new_deaths_smoothed',
     'Containment health index',
     
     ]]


dl = dl.dropna()
y = dl["pct_mask"] #output--target
X = dl.drop(columns=["pct_mask"])  #input--features


#train set e test set           20:80 split
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42) # random_state= seed
param_dist = {
    'max_depth': randint(5, 20),          # values from 5 to 20
    'min_samples_leaf': randint(1, 20),   # values from 1 to 20
    'max_features': ['sqrt', 'log2']
}
rf = RandomForestRegressor(n_estimators=300, random_state=42)
# RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=10,            # number of random combinations to test
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1             # use all available cores
)
random_search.fit(X, y) 
best_params = random_search.best_params_
print("Best params:", best_params)
#print("Best R²:", random_search.best_score_)


In [ ]:
# modello
rf = RandomForestRegressor(
    n_estimators=300,                
    max_depth=best_params['max_depth'],
    min_samples_leaf=best_params['min_samples_leaf'],
    max_features=best_params['max_features'],
    random_state=42,
    oob_score=True
)
rf.fit(X_train, y_train) # train model

# predizione
y_pred = rf.predict(X_test) # predict pcct_mask 

# valutazione del modello
mse = mean_squared_error(y_test, y_pred) # % error
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred) # explain variability
print(f"RMSE: {rmse:.4f}")
print(f"R2 Score: {r2:.4f}")

# Variables importance
importances = rf.feature_importances_ #feature_importances_
feature_names = X.columns
feat_importances = pd.Series(importances, index=feature_names)




# Predictions
y_pred_train = rf.predict(X_train)
y_pred_test = rf.predict(X_test)

# Metrics Train
r2_train = r2_score(y_train, y_pred_train)
mse_train = mean_squared_error(y_train, y_pred_train)

# Metrics Test
r2_test = r2_score(y_test, y_pred_test)
mse_test = mean_squared_error(y_test, y_pred_test)

# OOB score
oob_score = rf.oob_score_

print("=== PERFORMANCE RANDOM FOREST ===")
print(f"R² Train: {r2_train:.3f}")
print(f"R² Test:  {r2_test:.3f}")
print(f"MSE Train: {mse_train:.3f}")
print(f"MSE Test:  {mse_test:.3f}")
print(f"OOB Score: {oob_score:.3f}")



In [ ]:
palette = sns.color_palette("ch:start=.2,rot=-.3_r", n_colors=len(feat_importances.sort_values(ascending=False).index))

plt.figure(figsize=(7, 6))
ax = sns.barplot(x=feat_importances.sort_values(ascending=False), y=feat_importances.sort_values(ascending=False).index, hue=feat_importances.sort_values(ascending=False).index, palette=palette)
#plt.title("Importance of variables in pct_mask forecast", fontsize=16)
plt.xlabel("Importance", fontsize=15)
plt.ylabel("Variables", fontsize=15)
ax.set_yticks([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11])              # positions
ax.set_yticklabels(["Belief Masks Effective", "Worried Catch COVID (Bel)", "Containment Health Index", "Awa1 (Local health workers)", "Awa7 (Friends and family)", "New deaths", "Awa6 (Journalists)", "New cases","Awa4 (Govt. health authorities)", "Awa2 (Experts)", "Awa3 (WHO)", "Awa5 (Politicians)"])  # custom labels

plt.tight_layout()
plt.show()
print(feat_importances)


In [ ]:
from mlxtend.evaluate import bias_variance_decomp
from sklearn.ensemble import RandomForestRegressor
import numpy as np

# Bias-Variance decomposition
avg_expected_loss, avg_bias, avg_var = bias_variance_decomp(
    rf,        # model
    X_train.values, y_train.values,  # training data
    X_test.values, y_test.values,    # test data
    loss='mse',
    num_rounds=50,   # repetitions bootstrap
    random_seed=42
)

print(f"Bias^2: {avg_bias:.3f}, Variance: {avg_var:.3f}, MSE: {avg_expected_loss:.3f}")


In [ ]:
df = pd.read_csv('FULL_DATA5_FINALE.csv', parse_dates=['survey_date'])
selected_countries = ['Philippines', 'Greece', 'Canada', 'Argentina', 'Japan', 'Sweden']

# Modello Random Forest
#rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf = RandomForestRegressor(
    n_estimators=300,                
    max_depth=best_params['max_depth'],
    min_samples_leaf=best_params['min_samples_leaf'],
    max_features=best_params['max_features'],
    random_state=42,
    oob_score=True
)


In [ ]:
# Fig reconstruction (using all variables)

fig, axes = plt.subplots(len(selected_countries), 1, figsize=(7, 10), sharex=True)

for i, country in enumerate(selected_countries):
    df_country = df[df['country'] == country].copy()

    dl = df_country[['survey_date', 'pct_mask',
                     'pct_worried_catch_covid','pct_belief_masking_effective',
                     'pct_received_news_local_health','pct_received_news_experts','pct_received_news_who','pct_received_news_govt_health',
                     'pct_received_news_politicians','pct_received_news_journalists','pct_received_news_friends',
                     'new_cases_smoothed','new_deaths_smoothed','Containment health index']]

    dl = dl.dropna()

    date_col = dl['survey_date']
    y_all = dl['pct_mask']
    X_all = dl.drop(columns=['pct_mask', 'survey_date'])

    rf.fit(X_all, y_all)
    y_pred_all = rf.predict(X_all)

    results_df = pd.DataFrame({
        'survey_date': date_col,
        'true_mask': y_all,
        'predicted_mask': y_pred_all
    }).sort_values('survey_date')

    ax = axes[i]

    ax.plot(results_df['survey_date'], results_df['true_mask'], label='Real values', color='tab:blue', linewidth=1, alpha=0.8)
    ax.plot(results_df['survey_date'], results_df['predicted_mask'], label='Reconstructed with RF', color='tab:red', linewidth=1)

    ax.set_title(f'{country}', fontsize=14)
    ax.set_ylabel('', fontsize=14)
    axes[2].set_ylabel('Mask Wearing (% respondents)', fontsize=14)
    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.grid(True)

axes[-1].set_xlabel('Time', fontsize=14)  

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, fontsize=12, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=[0, 0.03, 1, 1]) 
plt.show()

In [ ]:
# Fig reconstruction (using only three top variables)

fig, axes = plt.subplots(len(selected_countries), 1, figsize=(7, 10), sharex=True)

for i, country in enumerate(selected_countries):
    df_country = df[df['country'] == country].copy()

    dl = df_country[['survey_date', 'pct_mask',
                     'pct_worried_catch_covid','pct_belief_masking_effective',
                     'Containment health index']]

    dl = dl.dropna()

    date_col = dl['survey_date']
    y_all = dl['pct_mask']
    X_all = dl.drop(columns=['pct_mask', 'survey_date'])

    rf.fit(X_all, y_all)
    y_pred_all = rf.predict(X_all)

    results_df = pd.DataFrame({
        'survey_date': date_col,
        'true_mask': y_all,
        'predicted_mask': y_pred_all
    }).sort_values('survey_date')

    ax = axes[i]

    ax.plot(results_df['survey_date'], results_df['true_mask'], label='Real values', color='tab:blue', linewidth=1, alpha=0.8)
    ax.plot(results_df['survey_date'], results_df['predicted_mask'], label='Reconstructed with RF', color='tab:red', linewidth=1)

    ax.set_title(f'{country}', fontsize=14)
    ax.set_ylabel('', fontsize=14)
    axes[2].set_ylabel('Mask Wearing (% respondents)', fontsize=14)
    ax.tick_params(axis='both', which='major', labelsize=12)
    ax.grid(True)

axes[-1].set_xlabel('Time', fontsize=14)  # etichetta asse x solo nell’ultimo subplot

# Legenda unica sotto tutti i subplot
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, fontsize=12, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=[0, 0.03, 1, 1]) 
plt.show()